# CellStateAdj on the WOT serum arm, days 0-18

Reads back what `03_fit.py` wrote and draws the time-layered DAG.
Nothing is fitted here -- run the stages first:

```bash
sbatch 01_eps_scan.sbatch                        # Stage A -> epsilon*
sbatch 02_k_sweep.sbatch --epsilon <eps*>        # Stage B (array over K)
python  02_k_reduce.py   --epsilon <eps*>        #          -> K*
sbatch 03_fit.sbatch --epsilon <eps*> --K <K*>   # Stage C
```

**This notebook does not judge whether the method works.** V+-, G+-, the degeneracy
verdict and the baseline comparison are loaded and shown, but interpreting them is a
separate pass.


In [ ]:
import json, os, sys, gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

sys.path.insert(0, os.path.abspath("."))
from csa_wot import RESULTS_ROOT

RUN = "fit_main"          # or "fit_lam0"
RUN_DIR = os.path.join(RESULTS_ROOT, RUN)
print(RUN_DIR)
print(sorted(os.listdir(RUN_DIR)))

## 1. Run summary and convergence

`line_search_failed` and `max_iter` are failures, not convergence.

In [ ]:
summary = json.load(open(os.path.join(RUN_DIR, "summary.json")))
print(f"status      : {summary['status']}  (converged={summary['converged']})")
print(f"iterations  : {summary['n_iter']}   wall time {summary['wall_time_s']/60:.1f} min")
print(f"objective   : {summary['objective']:.6f}")
print(f"epsilon / K : {summary['epsilon']} / {summary['K']}")
print(f"DAG         : {summary['dag']}")
print(f"mass check  : {summary['mass_conservation']}")
print()
print("terms:", json.dumps(summary["terms"], indent=2))
print()
print("CAVEAT:", summary["branch_caveat"])

## 2. Optimisation trace

The objective is not monotone by default -- L_+ at t depends on M_{t+1}, so this is self-referential clustering. Convergence is to a fixed point, not necessarily monotone descent.

In [ ]:
hist = pd.read_csv(os.path.join(RUN_DIR, "history.csv"))
fig, ax = plt.subplots(1, 4, figsize=(17, 3.4))
ax[0].plot(hist["iter"], hist["total"]);      ax[0].set_title("total objective")
for k in ("compress", "expression", "plus", "minus"):
    if k in hist and hist[k].abs().sum() > 0:
        ax[1].plot(hist["iter"], hist[k], label=k)
ax[1].legend(fontsize=8); ax[1].set_title("components")
ax[2].plot(hist["iter"], hist["k_eff_mean"], label="mean")
ax[2].plot(hist["iter"], hist["k_eff_min"], label="min")
ax[2].axhline(summary["K"], ls="--", c="k", lw=.8, label="K")
ax[2].legend(fontsize=8); ax[2].set_title("K_eff = exp(H(g_t))  [coarsening watch]")
ax[3].semilogy(hist["iter"], hist["dM"]); ax[3].set_title("||dM||")
for a in ax: a.set_xlabel("iteration")
plt.tight_layout(); plt.show()
hist.tail(3)

## 3. Load the matrices

Every optimised object is on disk: `P^ref` (sparse triplets), `M_t`, `T_t`, `A_t`, `B_t`, `g_t`, `mu_t`, the per-cell fingerprints `f+-` with their prototypes, and `V+-`/`G+-`.

In [ ]:
mem   = np.load(os.path.join(RUN_DIR, "memberships.npz"))
trans = np.load(os.path.join(RUN_DIR, "transitions.npz"))
diag  = np.load(os.path.join(RUN_DIR, "diagnostics.npz"))
fp    = np.load(os.path.join(RUN_DIR, "fingerprints.npz"))

tau = mem["tau"]; T = len(tau); K = mem["M_0"].shape[1]
M = [mem[f"M_{t}"] for t in range(T)]
g = [trans[f"g_{t}"] for t in range(T)]
Tm = [trans[f"T_{t}"] for t in range(T-1)]
A  = [trans[f"A_{t}"] for t in range(T-1)]
B  = [trans[f"B_{t}"] for t in range(T-1)]

print(f"T={T} timepoints, K={K} states, days {tau[0]:g}-{tau[-1]:g}")
assert all(np.allclose(m.sum(1), 1, atol=1e-5) for m in M), "M rows must sum to 1"
assert all(abs(t.sum() - 1) < 1e-5 for t in Tm), "each T_t must have total mass 1"
assert all(np.allclose(a.sum(1), 1, atol=1e-5) for a in A), "A_t must be row-stochastic"
print("shape checks passed")

cells_df = pd.read_csv(os.path.join(RUN_DIR, "cell_table.csv.gz"))
cells_df.head()

## 4. The DAG

Nodes are `(t, k)` above the mass floor; edges carry `T_t[k,l]`. Time-layered, so acyclic by construction. State index `k` is **time-local** -- `k` at `t` has no relation to `k` at `t+1`; the only linkage is transported mass.

Edges are transport-implied developmental compatibility, never observed lineage.

In [ ]:
G = nx.read_graphml(os.path.join(RUN_DIR, "dag.graphml"))
print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges, "
      f"acyclic={nx.is_directed_acyclic_graph(G)}")

# dominant annotation per (t, k), for colouring only
ann_col = "major_cell_sets" if "major_cell_sets" in cells_df else None
if ann_col:
    dom = (cells_df.groupby(["t", "state"])[ann_col]
           .agg(lambda s: s.value_counts().idxmax() if len(s) else "NA"))
    cats = sorted(cells_df[ann_col].dropna().unique())
    cmap = {c: plt.cm.tab10(i % 10) for i, c in enumerate(cats)}
else:
    dom, cmap = None, {}

In [ ]:
def draw_dag(G, min_edge_mass=1e-3, figsize=(19, 8)):
    """Day on x, states stacked by mass on y; node area ~ mass, edge width ~ mass."""
    pos, node_size, node_color, order = {}, [], [], {}
    for n, d in G.nodes(data=True):
        order.setdefault(d["t"], []).append((d["mass"], n, d))
    fig, ax = plt.subplots(figsize=figsize)
    for t, items in order.items():
        items.sort(reverse=True)
        for rank, (mass, n, d) in enumerate(items):
            pos[n] = (d["day"], rank)
    for n, d in G.nodes(data=True):
        node_size.append(40 + 4000 * d["mass"])
        lab = dom.get((d["t"], d["state"]), "NA") if dom is not None else "NA"
        node_color.append(cmap.get(lab, (.6, .6, .6, 1.)))
    keep = [e for e in G.edges if G.edges[e]["mass"] >= min_edge_mass]
    nx.draw_networkx_edges(G, pos, edgelist=keep, ax=ax, alpha=.35,
                           width=[12 * G.edges[e]["mass"] for e in keep],
                           arrows=False, edge_color="0.35")
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_size,
                           node_color=node_color, linewidths=.4, edgecolors="w")
    ax.set_xlabel("day"); ax.set_ylabel("state (ranked by mass within timepoint)")
    ax.set_title(f"{RUN}: transition-defined states, WOT serum arm "
                 f"(K={K}, eps={summary['epsilon']}, lambda_pm={summary['lambda_pm']})")
    if cmap:
        handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=k)
                   for k, c in cmap.items()]
        ax.legend(handles=handles, fontsize=8, ncol=2, loc="upper left",
                  title="dominant cell set (colour only)")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()

draw_dag(G)

## 5. State masses and the coarsening watch

`K_eff_t = exp(H(g_t))` well below `K` means the effective occupancy has collapsed even though `K` is fixed -- Degeneracy 3. That is a result to report, not only a bug.

In [ ]:
Gm = np.stack(g)                      # (T, K)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
im = ax[0].imshow(Gm.T, aspect="auto", origin="lower", cmap="magma",
                  extent=[tau[0], tau[-1], -.5, K-.5])
ax[0].set_xlabel("day"); ax[0].set_ylabel("state"); ax[0].set_title("state mass g_tk")
plt.colorbar(im, ax=ax[0])

keff = diag["k_eff"]
ax[1].plot(tau, keff, "o-")
ax[1].axhline(K, ls="--", c="k", lw=.8, label=f"K = {K}")
ax[1].set_xlabel("day"); ax[1].set_ylabel("K_eff = exp(H(g_t))")
ax[1].set_title("effective number of occupied states"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"K_eff: min {keff.min():.2f}  median {np.median(keff):.2f}  max {keff.max():.2f}")

## 6. Node table

`dag_nodes.csv` carries per-state mass, N_child / N_parent, V+-, G+- and the (V, G) quadrant. Loaded here; interpreting it is the next pass, not this one.

In [ ]:
nodes = pd.read_csv(os.path.join(RUN_DIR, "dag_nodes.csv"))
print(nodes["label"].value_counts().to_string())
print()
print(nodes["geometry_plus"].value_counts().to_string())
nodes.sort_values("mass", ascending=False).head(12)

## 7. Where the states sit in expression space

Composition of each state against the published `cell_sets` annotation. This is a description of the fit, not a validation of it.

In [ ]:
if ann_col:
    ct = pd.crosstab(cells_df["day"], cells_df[ann_col], normalize="index")
    fig, ax = plt.subplots(figsize=(13, 4))
    ct.plot.area(ax=ax, cmap="tab10", lw=0)
    ax.set_xlim(tau[0], tau[-1]); ax.set_ylabel("fraction of cells")
    ax.set_title("published annotation over time (context for the DAG)")
    ax.legend(fontsize=8, ncol=2, bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout(); plt.show()

    comp = pd.crosstab(cells_df["state"], cells_df[ann_col], normalize="index")
    display(comp.round(3))

## 8. lambda_pm = 0 vs lambda_pm > 0

How much the fingerprint terms actually moved the memberships. `manifest.json` holds the ARI and L1 change computed at the end of Stage C.

In [ ]:
man_path = os.path.join(RESULTS_ROOT, "manifest.json")
if os.path.exists(man_path):
    man = json.load(open(man_path))
    print(json.dumps(man, indent=2)[:2500])
else:
    print("no manifest.json -- run 03_fit.py with both --runs lam0 main")

## 9. Stage A and Stage B, for the record

In [ ]:
eps_path = os.path.join(RESULTS_ROOT, "eps_scan", "summary.json")
if os.path.exists(eps_path):
    eps = json.load(open(eps_path))
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
    for stride, blk in eps["per_stride"].items():
        e = np.asarray(eps["epsilons"], float)
        mc = blk["mean_curves"]
        ax[0].semilogx(e, mc["I_cell_normalized"], "o-", label=f"stride {stride}")
        ax[1].semilogx(e, mc["I_fingerprint_plus"], "o-", label=f"stride {stride}")
        ax[2].semilogx(e, mc["stability_resample"], "o-", label=f"stride {stride}")
        print(f"stride {stride}: {blk['recommendation']}")
    for a, t in zip(ax, ["I_cell normalized", "I_fingerprint_plus", "stability_resample"]):
        a.set_xlabel("epsilon"); a.set_title(t); a.legend(fontsize=8)
    plt.tight_layout(); plt.show()

k_path = os.path.join(RESULTS_ROOT, "k_sweep", "k_selection.json")
if os.path.exists(k_path):
    ks = json.load(open(k_path))
    kf = pd.DataFrame(ks["per_K"]).sort_values("K")
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].errorbar(kf["K"], kf["heldout_compress"], yerr=kf["heldout_se"], fmt="o-",
                   label="held out")
    ax[0].plot(kf["K"], kf["train_compress"], "s--", label="train")
    ax[0].set_xlabel("K"); ax[0].set_ylabel("L_compress"); ax[0].legend(fontsize=8)
    ax[1].semilogy(kf["K"], kf["min_state_mass"], "o-")
    ax[1].axhline(1e-3, ls="--", c="k", lw=.8); ax[1].set_xlabel("K")
    ax[1].set_ylabel("min state mass")
    plt.tight_layout(); plt.show()
    print(json.dumps(ks["recommendation"], indent=2))